Φόρτωση Μοντέλου: Helsinki-NLP/opus-mt-grk-en

https://huggingface.co/Helsinki-NLP/opus-mt-grk-en

In [1]:
!pip install -q transformers sentencepiece sacrebleu sacremoses

In [2]:
from transformers import MarianMTModel, MarianTokenizer
import torch

In [3]:
model_name = "Helsinki-NLP/opus-mt-grk-en"

tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"

model = model.to(device)

tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/507k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/441k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/802k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  225MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  224MB            

model.safetensors: downloading bytes:           |  0.00B            

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

Δοκιμή

In [4]:
text = "Καλημέρα, σήμερα είναι Σάββατο και ο καιρός είναι ζεστός."

inputs = tokenizer(text, return_tensors = "pt", padding = True).to(device)
translated = model.generate(**inputs)

print(tokenizer.batch_decode(translated, skip_special_tokens=True)[0])

Good morning, it's Saturday and the weather is warm.


Εκτέλεση στο Flores Dataset

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
import json

In [7]:
# Φόρτωση των δεδομένων

with open("/content/drive/MyDrive/Μηχανική Μετάφραση_Εργασία 1/Flores_ell.json", "r", encoding = "utf-8") as f:
  ell_dataset = json.load(f)

with open("/content/drive/MyDrive/Μηχανική Μετάφραση_Εργασία 1/Flores_eng.json", "r", encoding = "utf-8") as f:
  eng_dataset = json.load(f)

In [8]:
len(ell_dataset), len(eng_dataset)

(1012, 1012)

In [9]:
# Τρέχουμε στο ell_dataset

batch_size = 32
model_translations = []

In [10]:
for i in range(0, len(ell_dataset), batch_size):
  batch = ell_dataset[i : i + batch_size]

  inputs = tokenizer(batch, padding=True, truncation=True, return_tensors="pt").to(device)

  translations = model.generate(**inputs,max_length=256)

  batch_preds = tokenizer.batch_decode(translations, skip_special_tokens=True)
  model_translations.extend(batch_preds)

In [11]:
len(model_translations)

1012

Αποθήκευση των μεταφράσεων του μοντέλου

In [12]:
import os

In [13]:
filename = "Helsinki_NLP_Translations" + ".json"
path = os.path.join(f"/content/drive/MyDrive/Μηχανική Μετάφραση_Εργασία 1", filename)

with open(path, "w", encoding="utf-8") as f:
    json.dump(model_translations, f, ensure_ascii=False, indent=4)

Αξιολόγηση Μοντέλου

In [14]:
!pip install -q evaluate sacrebleu

In [15]:

import evaluate

In [16]:
BLEU= evaluate.load("sacrebleu")
CHRF = evaluate.load("chrf")

BLEU_results = BLEU.compute(predictions = model_translations, references = [[ref] for ref in eng_dataset])
CHRF_results = CHRF.compute(predictions = model_translations, references = [[ref] for ref in eng_dataset])

print("BLEU:", BLEU_results["score"])
print("chrF:", CHRF_results["score"])

BLEU: 27.383005815157855
chrF: 56.11597367050298
